# 3. SPRINT 2 — Preparación de datos y modelado inicial

**Proyecto de investigación:** *Modelo predictivo de horas de indisponibilidad operativa mediante Gradient Boosting Machine (GBM) considerando variables meteorológicas y oceanográficas en el Terminal Portuario ENGIE Perú, Ilo.*

**Marco metodológico:** CRISP-DM (fase *Preparación de datos*) · **Marco de gestión:** Scrum · **Sprint:** Sprint 2 — Semanas 7 a 11

**Estructura de esta parte del notebook**

| N.° | Historia de usuario / Tarea | Contenido |
|---|---|---|
| 3.1 | **HU05** — Limpieza del dataset | |
| 3.1.1 | T09 | Tratar valores nulos y duplicados (**5 story points**) |
| 3.1.2 | T10 | Corregir inconsistencias en horas de indisponibilidad (**3 story points**) |

> **Insumos (salidas del Sprint 1):** `dataset_integrado_horario_HU03.csv` (malla horaria integrada) y `Historico_Indisponibilidad_Operativa*.csv` (eventos con `fecha_inicio` / `fecha_fin`). Opcionalmente, los CSV crudos del datalogger, para re-verificar duplicados por marca de tiempo.
>
> **Salidas de HU05 (insumo de HU06 – Ingeniería de características):** dataset horario limpio + banderas de imputación + eventos depurados + bitácoras de decisiones.

## 3.1 HU05 — Limpieza del dataset

**Historia de usuario:** *Como Científico de datos, quiero depurar los registros históricos, para mejorar la calidad de los datos utilizados por el modelo.*

### Criterios de aceptación de HU05 (TB1)

1. Se aplica y documenta el tratamiento de **nulos, duplicados e inconsistencias**.
2. Se alcanza una **completitud ≥ 90 %** o se **justifica** el tratamiento de los registros faltantes.
3. Los vacíos se tratan **de forma diferenciada por variable** (la tolerancia a la imputación no es igual para viento que para oleaje).
4. Cada decisión (imputar, excluir, corregir) queda **trazable** en una bitácora.

### Regla de tratamiento de faltantes (inciso f del pipeline)

| Tipo de vacío | Causa típica | Tratamiento |
|---|---|---|
| **Puntual** (≤ límite de horas de la variable) | falla breve de sensor | **Interpolación temporal** (lineal o spline), acotada al máximo de horas continuas |
| **Prolongado** (> límite) | mantenimiento o falla de estación | **No se imputa**: se documenta y se evalúa excluir el bloque temporal o la variable |

Antes de remuestrear/agregar por hora, se verifica y **deduplica por marca de tiempo**, conservando la **última lectura válida**. En el Sprint 1 este paso se ejecutó dentro del remuestreo y no detectó duplicados en ninguna de las ocho series; aquí se **re-verifica de forma explícita y documentada**.

In [ ]:
# ==== HU05 - Preparación del entorno (estilo y funciones de dibujo compartidas) ====
import glob, re, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import display

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 50)

COLOR_NAVY    = '#1B3A5C'
COLOR_TEAL    = '#2A9D8F'
COLOR_TEAL_LT = '#D6EFEC'
COLOR_ORANGE  = '#E9762B'
COLOR_RED     = '#C43D3D'
COLOR_GRAY    = '#9AA5B1'
COLOR_GRAY_LT = '#EEF1F4'
COLOR_GOLD    = '#E3A857'
plt.rcParams['font.family'] = 'DejaVu Sans'

def caja(ax, x, y, w, h, texto, color=COLOR_NAVY, texto_color='white', fontsize=10, fontweight='bold'):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.02,rounding_size=0.05',
                                linewidth=1.5, edgecolor=color, facecolor=color, zorder=2))
    ax.text(x + w/2, y + h/2, texto, ha='center', va='center', fontsize=fontsize,
            color=texto_color, fontweight=fontweight, zorder=3, linespacing=1.35)

def flecha(ax, p1, p2, color=COLOR_NAVY, lw=1.8, rad=0.0):
    ax.add_patch(FancyArrowPatch(p1, p2, arrowstyle='-|>', mutation_scale=15, color=color,
                                 lw=lw, connectionstyle=f'arc3,rad={rad}', zorder=1))

def lienzo(w=11, h=5, xlim=(0, 11), ylim=(0, 5)):
    fig, ax = plt.subplots(figsize=(w, h))
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.axis('off')
    return fig, ax

MPH_A_MS = 0.44704
print('Entorno listo.')

### Vista rápida: el flujo completo de HU05

```
T09 (nulos y duplicados) ─────────────────────────────►  T10 (inconsistencias en horas)
Serie horaria: dedup → rango físico → vacíos → imputar    Eventos: nulos → duplicados → fechas → duración
```

In [ ]:
fig, ax = lienzo(13, 5.2, xlim=(0, 13), ylim=(0, 5.4))
ax.set_title('HU05 — Limpieza del dataset: de los insumos del Sprint 1 al dataset entrenable',
             fontsize=13, color=COLOR_NAVY, fontweight='bold', pad=12)

caja(ax, 0.2, 3.6, 2.3, 1.1, 'Entrada\nDataset horario HU03\n+ histórico de eventos', color=COLOR_GRAY, fontsize=8.8)
flecha(ax, (2.5, 4.15), (3.0, 4.15))

# Carril T09
ax.text(3.0, 5.15, 'T09 · Nulos y duplicados (series horarias)', fontsize=9.5, color=COLOR_TEAL, fontweight='bold')
pasos9 = ['1. Verificar y\ndeduplicar por\nmarca de tiempo', '2. Rango físico\n(valores imposibles\n→ nulo)', '3. Diagnóstico\nde vacíos:\npuntual vs prolongado',
          '4. Interpolación\nlineal / spline /\ncircular acotada']
for i, p in enumerate(pasos9):
    x = 3.0 + i * 2.5
    caja(ax, x, 3.6, 2.2, 1.1, p, color=COLOR_TEAL, fontsize=8.2)
    if i < 3: flecha(ax, (x + 2.2, 4.15), (x + 2.5, 4.15), color=COLOR_TEAL)

# Carril T10
ax.text(3.0, 2.85, 'T10 · Inconsistencias en horas de indisponibilidad (eventos)', fontsize=9.5, color=COLOR_ORANGE, fontweight='bold')
pasos10 = ['5. Nulos y\nduplicados\nen eventos', '6. Fechas\ninvertidas y\nduración ≠ fin−inicio', '7. Formato, solapes\ny causa no\nmeteo-oceanográfica']
for i, p in enumerate(pasos10):
    x = 3.0 + i * 2.5
    caja(ax, x, 1.4, 2.2, 1.1, p, color=COLOR_ORANGE, fontsize=8.2)
    if i < 2: flecha(ax, (x + 2.2, 1.95), (x + 2.5, 1.95), color=COLOR_ORANGE)

flecha(ax, (11.6, 3.6), (11.6, 2.5), color=COLOR_TEAL)
caja(ax, 10.4, 1.4, 2.4, 1.1, 'Targets rehechos\ncon eventos\nlimpios', color=COLOR_NAVY, fontsize=8.5)
flecha(ax, (8.0, 1.95), (10.4, 1.95), color=COLOR_ORANGE)

flecha(ax, (11.6, 1.4), (11.6, 0.95), color=COLOR_NAVY)
caja(ax, 3.0, 0.15, 9.8, 0.8, 'Salida → HU06: dataset limpio + banderas de imputación + eventos depurados + bitácoras de decisión', color=COLOR_NAVY, fontsize=9.3)
plt.tight_layout(); plt.show()

### Configuración de HU05 (parámetros editables)

Todo lo que es **criterio de negocio** vive en una sola celda, para que pueda **validarse con la asesora y el Product Owner** y re-ejecutarse sin tocar el código.

> **Nota:** los límites de horas (`max_gap_h`) y los rangos físicos son **valores de partida razonables, no resultados medidos**. Se contrastan con el diagnóstico de vacíos (T09.3) y se ajustan si la asesora o Operaciones lo indican.

**Lógica de la tolerancia diferenciada por variable**
- **Viento (velocidad y dirección)** y **oleaje**: cambian rápido (ráfagas, grupos de olas) → límite **corto**, porque interpolar horas largas inventaría condiciones que pudieron ser justo las críticas para la operación.
- **Temperatura, presión y humedad**: variables **suaves** → límite **más amplio**; la temperatura y la presión usan **spline cúbico local** por su evolución continua.
- **Dirección del viento**: se interpola de forma **circular** (componentes seno/coseno), porque interpolar grados directamente falla en el salto 360° → 0°.
- **Diagnóstico del datalogger** (temperatura interna, voltaje): no son predictores; se tratan para conservar la trazabilidad y explicar vacíos compartidos.

In [ ]:
# ==== HU05 - Configuración (EDITABLE) ====
# rol: 'predictor' entra al modelo; 'diagnostico' solo explica fallas del datalogger
# metodo: 'linear' | 'spline' (cúbico local) | 'circular' (grados, vía seno/coseno)
# max_gap_h: máximo de horas CONSECUTIVAS sin dato que se interpolan (vacío puntual)
# rango: valores físicamente posibles; fuera de rango -> se convierte en nulo antes de imputar
# max_pct_vacio: si tras imputar queda más de este % de vacío, se evalúa excluir la variable
CONFIG_VARIABLES = {
    'velocidad_viento_ms':              dict(rol='predictor',   metodo='linear',   max_gap_h=3, rango=(0, 60),    max_pct_vacio=10),
    'direccion_viento_deg':             dict(rol='predictor',   metodo='circular', max_gap_h=3, rango=(0, 360),   max_pct_vacio=10),
    'temperatura_c':                    dict(rol='predictor',   metodo='spline',   max_gap_h=6, rango=(-5, 45),   max_pct_vacio=10),
    'humedad_relativa_pct':             dict(rol='predictor',   metodo='linear',   max_gap_h=6, rango=(0, 100),   max_pct_vacio=10),
    'presion_atmosferica_hpa':          dict(rol='predictor',   metodo='spline',   max_gap_h=6, rango=(900, 1100), max_pct_vacio=10),
    'altura_olas_m':                    dict(rol='predictor',   metodo='linear',   max_gap_h=2, rango=(0, 15),    max_pct_vacio=10),
    'temperatura_interna_datalogger_c': dict(rol='diagnostico', metodo='linear',   max_gap_h=6, rango=(-10, 80),  max_pct_vacio=10),
    'voltaje_datalogger_v':             dict(rol='diagnostico', metodo='linear',   max_gap_h=6, rango=(0, 30),    max_pct_vacio=10),
}
VARIABLES = list(CONFIG_VARIABLES)
PREDICTORES = [v for v, c in CONFIG_VARIABLES.items() if c['rol'] == 'predictor']
DIAGNOSTICO = [v for v, c in CONFIG_VARIABLES.items() if c['rol'] == 'diagnostico']

UMBRAL_COMPLETITUD_PCT = 90          # criterio de aceptación HU05 (TB1)
VENTANA_SPLINE_H = 12                # puntos válidos a cada lado del hueco para el spline local
HORIZONTE_H = 24                     # horizonte de predicción (HU01/T02)

# ---- T10: reglas sobre el histórico de eventos ----
TOL_DURACION_H = 0.05                # tolerancia entre duración registrada y (fin - inicio)
UMBRAL_REVISION_H = 1.0              # diferencias mayores a esto se marcan para validar con Operaciones
POLITICA_FECHA_FIN_NULA = 'imputar_con_duracion_registrada'  # o 'excluir'. Imputa solo si duracion_h_registrada y duracion_h_calculada coinciden
FUSIONAR_SOLAPADOS = False           # False: solo marca solapes; True: fusiona eventos solapados
COLUMNAS_CAUSA_CANDIDATAS = ['causa_registrada', 'causa', 'causa_indisponibilidad', 'motivo', 'tipo_causa']
PALABRAS_CAUSA_NO_CLIMATICA = ['mantenim', 'falla', 'electric', 'energ', 'averia', 'mecanic', 'programad', 'comunicacion', 'corte de']

display(pd.DataFrame(CONFIG_VARIABLES).T.rename(columns={
    'rol': 'Rol', 'metodo': 'Método', 'max_gap_h': 'Máx. horas a interpolar',
    'rango': 'Rango físico', 'max_pct_vacio': '% vacío máx. tolerado'}))

### Carga de los insumos del Sprint 1

Se leen `dataset_integrado_horario_HU03.csv` y el histórico de eventos. Si no están en la sesión de Colab, la celda ofrece subirlos. **Las columnas de targets provisionales de HU03 se descartan**: se reconstruirán al final con los eventos ya limpios.

In [ ]:
# ==== Carga de insumos ====
RUTA_DATASET = 'dataset_integrado_horario_HU03.csv'
PATRON_EVENTOS = 'Historico_Indisponibilidad_Operativa*.csv'

faltantes = [p for p in [RUTA_DATASET, PATRON_EVENTOS] if not glob.glob(p)]
if faltantes:
    try:
        from google.colab import files
        print('Faltan archivos; selecciona los CSV a subir:', *faltantes, sep='\n  - ')
        files.upload()
    except ImportError:
        raise FileNotFoundError(f'Faltan archivos en el directorio de trabajo: {faltantes}')

df_hu03 = pd.read_csv(RUTA_DATASET, index_col='fecha_hora', parse_dates=['fecha_hora'])
ruta_eventos = glob.glob(PATRON_EVENTOS)[0]
df_eventos_crudo = pd.read_csv(ruta_eventos, parse_dates=['fecha_inicio', 'fecha_fin'])

# Validación mínima de esquema
faltan_vars = [v for v in VARIABLES if v not in df_hu03.columns]
faltan_cols_ev = [c for c in ['id_evento', 'fecha_inicio', 'fecha_fin'] if c not in df_eventos_crudo.columns]
assert not faltan_vars, f'Faltan variables en el dataset HU03: {faltan_vars}'
assert not faltan_cols_ev, f'Faltan columnas en el histórico de eventos: {faltan_cols_ev}'

print(f'Dataset HU03 : {df_hu03.shape[0]:,} filas x {df_hu03.shape[1]} columnas  ({df_hu03.index.min()} -> {df_hu03.index.max()})')
print(f'Histórico    : {len(df_eventos_crudo):,} eventos  (archivo: {Path(ruta_eventos).name})')
display(df_eventos_crudo.head(3))

## 3.1.1 T09 — Tratar valores nulos y duplicados

### T09.1 Verificación y deduplicación por marca de tiempo (paso previo a la agregación horaria)

Se verifican **tres niveles**:

1. **CSV crudos** (si están en la sesión): marcas de tiempo repetidas por variable. Si una marca se repite se conserva la **última lectura válida** (no nula).
2. **Índice de la malla horaria**: no debe haber horas repetidas ni saltos.
3. **Tabla resultante**: confirma o refuta el hallazgo del Sprint 1 (*0 duplicados en las ocho series*).

In [ ]:
# ==== T09.1 - Verificación y deduplicación por marca de tiempo ====
PATRONES_CRUDOS = {
    'velocidad_viento_mph':              'Velocidad de Viento*.csv',
    'direccion_viento_deg':              'Direccion de Viento*.csv',
    'temperatura_c':                     'Temperatura-*.csv',
    'humedad_relativa_pct':              'Humedad Relativa*.csv',
    'presion_atmosferica_hpa':           'Presion Atmosferica*.csv',
    'altura_olas_m':                     'Altura de Olas*.csv',
    'temperatura_interna_datalogger_c':  'Temperatura Interna de Datalogger*.csv',
    'voltaje_datalogger_v':              'Voltaje de Datalogger*.csv',
}

def _a_numero(v):
    if pd.isna(v): return np.nan
    m = re.search(r'-?\d+(\.\d+)?', str(v).strip().replace(',', '.'))
    return float(m.group()) if m else np.nan

def deduplicar_ultima_lectura_valida(df):
    """Ordena por marca de tiempo y, si una marca se repite, conserva la ULTIMA lectura VALIDA
    (no nula). Si todas las lecturas repetidas son nulas, conserva una fila nula."""
    df = df.dropna(subset=['fecha_hora']).copy()
    df['_orden'] = np.arange(len(df))
    df['_valida'] = df['valor'].notna()
    df = df.sort_values(['fecha_hora', '_valida', '_orden'])      # las válidas quedan al final de cada marca
    return df.drop_duplicates('fecha_hora', keep='last').drop(columns=['_orden', '_valida'])

def verificar_duplicados_crudo(columna, patron):
    rutas = glob.glob(patron)
    if not rutas:
        return {'serie': columna, 'fuente': 'CSV crudo no disponible', 'lecturas': np.nan,
                'marcas_repetidas': np.nan, 'con_valores_distintos': np.nan, 'lecturas_tras_dedup': np.nan}
    df = pd.read_csv(rutas[0], encoding='utf-8')
    df = df.iloc[:, :2]; df.columns = ['Time', 'crudo']
    df['fecha_hora'] = pd.to_datetime(df['Time'], errors='coerce')
    df['valor'] = df['crudo'].apply(_a_numero)
    rep = df.dropna(subset=['fecha_hora']).duplicated('fecha_hora', keep=False)
    n_rep = int(df.dropna(subset=['fecha_hora']).duplicated('fecha_hora').sum())
    n_conf = int(df.dropna(subset=['fecha_hora'])[rep].groupby('fecha_hora')['valor'].nunique(dropna=True).gt(1).sum()) if rep.any() else 0
    dedup = deduplicar_ultima_lectura_valida(df[['fecha_hora', 'valor']])
    return {'serie': columna, 'fuente': Path(rutas[0]).name, 'lecturas': len(df),
            'marcas_repetidas': n_rep, 'con_valores_distintos': n_conf, 'lecturas_tras_dedup': len(dedup)}

reporte_dup_crudo = pd.DataFrame([verificar_duplicados_crudo(c, p) for c, p in PATRONES_CRUDOS.items()])
display(reporte_dup_crudo)

# --- Nivel 2: índice de la malla horaria
indice_duplicado = int(df_hu03.index.duplicated().sum())
df_hu03 = df_hu03[~df_hu03.index.duplicated(keep='last')].sort_index()
malla = pd.date_range(df_hu03.index.min(), df_hu03.index.max(), freq='h')
horas_faltantes_en_malla = len(malla) - len(df_hu03)
df_h = df_hu03.reindex(malla)
df_h.index.name = 'fecha_hora'

duplicados_total_crudo = reporte_dup_crudo['marcas_repetidas'].sum(skipna=True)
resumen_dup = pd.DataFrame([
    ['Series crudas verificadas', int(reporte_dup_crudo['lecturas'].notna().sum()), 'de 8'],
    ['Marcas de tiempo repetidas en CSV crudos', int(duplicados_total_crudo), 'Sprint 1 reportó 0 en las 8 series'],
    ['Horas repetidas en el índice horario', indice_duplicado, 'se conserva la última'],
    ['Horas ausentes en la malla (se reinsertan como nulo)', int(horas_faltantes_en_malla), 'malla horaria continua'],
], columns=['Verificación', 'Resultado', 'Comentario'])
display(resumen_dup)

### T09.2 Valores físicamente imposibles → nulo

Antes de medir vacíos, cualquier valor fuera del **rango físico** configurado se convierte en nulo y **se cuenta**. Es parte del tratamiento de nulos (un valor imposible equivale a una lectura inexistente). **No** se tocan los *outliers* estadísticos (IQR) de T08: son valores posibles, y por ejemplo el 8 % de `voltaje_datalogger_v` es un hallazgo a documentar, no a corregir.

In [ ]:
# ==== T09.2 - Rango físico ====
df_crudo_h = df_h[VARIABLES].copy()          # copia previa a cualquier tratamiento (para el "antes")
df_rango = df_crudo_h.copy()
filas = []
for v, cfg in CONFIG_VARIABLES.items():
    lo, hi = cfg['rango']
    fuera = (df_rango[v] < lo) | (df_rango[v] > hi)
    filas.append({'variable': v, 'rango_fisico': f'[{lo}, {hi}]', 'valores_fuera_de_rango': int(fuera.sum())})
    df_rango.loc[fuera, v] = np.nan
reporte_rango = pd.DataFrame(filas)
display(reporte_rango)

### T09.3 Diagnóstico de vacíos: puntuales vs prolongados

Se construye una **bitácora de vacíos** (una fila por hueco: variable, inicio, fin, horas, posición) y cada hueco se clasifica con el límite **de su propia variable**:

- `puntual`: horas consecutivas ≤ `max_gap_h` → **se interpola**.
- `prolongado`: horas consecutivas > `max_gap_h` → **no se interpola**, se documenta.
- Los huecos en el **borde** de la serie (inicio o final) no tienen dato a ambos lados, por lo que **nunca** se interpolan.

In [ ]:
# ==== T09.3 - Bitácora de vacíos ====
def listar_vacios(serie, max_gap_h):
    es_nulo = serie.isna()
    grupo = (es_nulo != es_nulo.shift(fill_value=False)).cumsum()
    filas = []
    for _, bloque in serie.groupby(grupo):
        if not bloque.isna().all():
            continue
        ini, fin, n = bloque.index[0], bloque.index[-1], len(bloque)
        if ini == serie.index[0]:   pos = 'borde_inicial'
        elif fin == serie.index[-1]: pos = 'borde_final'
        else:                        pos = 'interno'
        if pos != 'interno':         clase = 'prolongado (borde)' if n > max_gap_h else 'puntual (borde, no interpolable)'
        else:                        clase = 'puntual' if n <= max_gap_h else 'prolongado'
        filas.append({'variable': serie.name, 'inicio': ini, 'fin': fin, 'horas': n, 'posicion': pos,
                      'max_gap_h_variable': max_gap_h, 'clasificacion': clase})
    return filas

bitacora_vacios = pd.DataFrame(
    [f for v in VARIABLES for f in listar_vacios(df_rango[v], CONFIG_VARIABLES[v]['max_gap_h'])]
)

if bitacora_vacios.empty:
    print('No se detectaron vacíos en ninguna variable.')
else:
    resumen_vacios = (bitacora_vacios
        .pivot_table(index='variable', columns='clasificacion', values='horas', aggfunc=['count', 'sum'], fill_value=0))
    resumen_vacios.columns = [f'{a}_{b}'.replace('count', 'n_huecos').replace('sum', 'horas') for a, b in resumen_vacios.columns]
    display(resumen_vacios)
    print('Huecos más largos por variable:')
    display(bitacora_vacios.sort_values('horas', ascending=False).groupby('variable').head(2).reset_index(drop=True))

In [ ]:
# ==== Vista visual - Mapa de disponibilidad (antes de imputar) ====
disp = (df_rango[VARIABLES].isna().resample('D').mean() * 100).T       # % de horas vacías por día
fig, ax = plt.subplots(figsize=(14, 4.2))
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('disp', ['white', COLOR_GOLD, COLOR_RED])
from matplotlib.colors import PowerNorm
im = ax.imshow(disp.values, aspect='auto', cmap=cmap, norm=PowerNorm(gamma=0.4, vmin=0, vmax=100),
               extent=[mdates.date2num(disp.columns[0]), mdates.date2num(disp.columns[-1]), len(VARIABLES), 0])
ax.xaxis_date(); ax.xaxis.set_major_formatter(mdates.DateFormatter('%b-%y'))
ax.set_yticks(np.arange(len(VARIABLES)) + 0.5); ax.set_yticklabels(VARIABLES, fontsize=8.5)
ax.set_title('T09.3 — ¿Cuándo faltan datos? % de horas vacías por día y variable (antes de imputar)',
             fontsize=12, color=COLOR_NAVY, fontweight='bold')
fig.colorbar(im, ax=ax, shrink=0.85, label='% de horas vacías en el día (escala realzada)')
fig.tight_layout(); plt.show()
print('Lectura: franjas verticales que cruzan TODAS las filas = falla de estación/datalogger; '
      'marcas aisladas en una fila = falla de un sensor.')

In [ ]:
# ==== Vista visual - ¿Qué tan largos son los huecos vs el límite de cada variable? ====
if not bitacora_vacios.empty:
    fig, axes = plt.subplots(2, 4, figsize=(16, 6.6))
    for ax, v in zip(axes.ravel(), VARIABLES):
        h = bitacora_vacios.loc[bitacora_vacios['variable'] == v, 'horas']
        lim = CONFIG_VARIABLES[v]['max_gap_h']
        if len(h):
            tope = max(h.max(), lim + 1)
            ax.hist(h.clip(upper=48), bins=np.arange(0.5, min(tope, 48) + 1.5, 1),
                    color=[COLOR_TEAL], edgecolor='white')
        ax.axvline(lim + 0.5, color=COLOR_RED, lw=1.5, ls='--', label=f'límite = {lim} h')
        ax.set_title(v, fontsize=9, color=COLOR_NAVY, fontweight='bold')
        ax.set_xlabel('horas consecutivas sin dato (tope 48)', fontsize=8)
        ax.legend(fontsize=7.5, frameon=False); ax.spines[['top', 'right']].set_visible(False)
    fig.suptitle('Huecos a la izquierda de la línea roja se interpolan; a la derecha se documentan',
                 fontsize=12.5, color=COLOR_NAVY, fontweight='bold')
    fig.tight_layout(); plt.show()

### T09.4 Interpolación temporal acotada (lineal / spline / circular)

Solo se imputan los **vacíos puntuales internos**. La función respeta el límite **por hueco completo**: un hueco de 5 h con límite de 3 h **no se rellena parcialmente** (a diferencia del parámetro `limit` de `pandas.interpolate`, que sí rellenaría las primeras 3 horas).

| Método | Cómo funciona | Variables |
|---|---|---|
| `linear` | recta entre el último y el primer dato válido que rodean el hueco | viento (m/s), humedad, oleaje, diagnóstico |
| `spline` | spline cúbico **local** (hasta 12 puntos válidos a cada lado); si no hay suficientes puntos, cae a lineal | temperatura, presión |
| `circular` | interpola seno y coseno por separado y recompone el ángulo | dirección del viento |

Tras interpolar, el resultado se **recorta al rango físico** de la variable (el spline puede sobrepasarlo). Cada hora imputada queda marcada en una **bandera** para que HU06/HU07 puedan distinguir dato medido de dato imputado.

In [ ]:
# ==== T09.4 - Imputación temporal acotada ====
from scipy.interpolate import CubicSpline

def _huecos_internos(serie):
    """Devuelve lista de (pos_ini, pos_fin) de los huecos de NaN que NO están en los bordes."""
    es_nulo = serie.isna().values
    n, i, out = len(es_nulo), 0, []
    while i < n:
        if es_nulo[i]:
            j = i
            while j + 1 < n and es_nulo[j + 1]:
                j += 1
            if i > 0 and j < n - 1:
                out.append((i, j))
            i = j + 1
        else:
            i += 1
    return out

def _lineal_en_hueco(valores, a, b):
    x0, x1 = a - 1, b + 1
    return np.interp(np.arange(a, b + 1), [x0, x1], [valores[x0], valores[x1]])

def _spline_en_hueco(valores, a, b, ventana):
    n = len(valores)
    valida = ~np.isnan(valores)
    izq = np.where(valida[max(0, a - ventana):a])[0] + max(0, a - ventana)
    der = np.where(valida[b + 1:min(n, b + 1 + ventana)])[0] + b + 1
    if len(izq) < 2 or len(der) < 2:
        return None
    xs = np.concatenate([izq, der])
    cs = CubicSpline(xs, valores[xs], bc_type='natural')
    return cs(np.arange(a, b + 1))

def imputar_acotado(serie, metodo, max_gap_h, rango, ventana=VENTANA_SPLINE_H):
    """Imputa SOLO huecos internos con longitud <= max_gap_h. Retorna (serie_imputada, mascara_imputado, metodo_efectivo_por_hueco)."""
    s = serie.copy()
    mask = pd.Series(False, index=s.index)
    usados = []
    if metodo == 'circular':
        rad = np.deg2rad(s)
        comp = {'sen': np.sin(rad), 'cos': np.cos(rad)}
        comp_imp = {}
        for k, c in comp.items():
            c_imp, m, _ = imputar_acotado(c, 'linear', max_gap_h, (-1, 1), ventana)
            comp_imp[k] = c_imp
            mask = mask | m
        ang = np.degrees(np.arctan2(comp_imp['sen'], comp_imp['cos'])) % 360
        s = s.where(~mask, ang)
        n_imp = int(((mask != mask.shift(fill_value=False)) & mask).sum())
        return s, mask, ['circular'] * n_imp
    valores = s.values.astype(float).copy()
    for a, b in _huecos_internos(serie):
        largo = b - a + 1
        if largo > max_gap_h:
            continue
        rel = None
        if metodo == 'spline':
            rel = _spline_en_hueco(valores, a, b, ventana)
            usados.append('spline' if rel is not None else 'lineal (respaldo)')
        if rel is None:
            rel = _lineal_en_hueco(valores, a, b)
            if metodo != 'spline': usados.append('lineal')
        valores[a:b + 1] = rel
        mask.iloc[a:b + 1] = True
    s = pd.Series(valores, index=serie.index, name=serie.name)
    lo, hi = rango
    s = s.where(~mask, s.clip(lo, hi))
    return s, mask, usados

df_limpio = df_rango.copy()
flags_imputacion = pd.DataFrame(index=df_limpio.index)
detalle_metodos = []

for v, cfg in CONFIG_VARIABLES.items():
    # La velocidad en mph NO se interpola aparte: se reconstruye desde m/s (evita inconsistencias entre columnas)
    s_imp, m_imp, usados = imputar_acotado(df_rango[v], cfg['metodo'], cfg['max_gap_h'], cfg['rango'])
    df_limpio[v] = s_imp
    flags_imputacion[f'{v}_imputado'] = m_imp
    detalle_metodos.append({'variable': v, 'metodo_configurado': cfg['metodo'], 'max_gap_h': cfg['max_gap_h'],
                            'huecos_imputados': len(usados), 'horas_imputadas': int(m_imp.sum()),
                            'metodo_efectivo': ', '.join(sorted(set(usados))) or '—'})

# Variable derivada en mph, coherente con m/s limpio
if 'velocidad_viento_mph' in df_h.columns:
    df_limpio['velocidad_viento_mph'] = df_limpio['velocidad_viento_ms'] / MPH_A_MS

resumen_imputacion = pd.DataFrame(detalle_metodos)
display(resumen_imputacion)

In [ ]:
# ==== Vista visual - Ejemplos de imputación (el hueco imputado más largo de cada método) ====
def graficar_ejemplo(ax, v, margen_h=18):
    m = flags_imputacion[f'{v}_imputado']
    if not m.any():
        ax.text(0.5, 0.5, f'{v}\n(sin huecos imputados)', ha='center', va='center', transform=ax.transAxes, color=COLOR_GRAY)
        ax.axis('off'); return
    grupo = (m != m.shift(fill_value=False)).cumsum()
    largos = m.groupby(grupo).sum()
    g_max = largos[largos == largos.max()].index[0]
    idx = m[(grupo == g_max) & m].index
    ini, fin = idx[0] - pd.Timedelta(hours=margen_h), idx[-1] + pd.Timedelta(hours=margen_h)
    ax.plot(df_rango.loc[ini:fin, v], color=COLOR_NAVY, lw=1.6, marker='o', ms=3, label='medido')
    ax.plot(df_limpio.loc[idx, v], color=COLOR_ORANGE, lw=0, marker='o', ms=6, label='imputado')
    ax.axvspan(idx[0], idx[-1], color=COLOR_GOLD, alpha=0.25, lw=0)
    ax.set_title(f'{v}  ·  {CONFIG_VARIABLES[v]["metodo"]}  ·  hueco de {len(idx)} h', fontsize=9.5,
                 color=COLOR_NAVY, fontweight='bold')
    ax.legend(fontsize=8, frameon=False); ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(axis='x', labelsize=7.5, rotation=20)

ejemplos = ['velocidad_viento_ms', 'direccion_viento_deg', 'temperatura_c', 'altura_olas_m']
fig, axes = plt.subplots(1, 4, figsize=(18, 3.9))
for ax, v in zip(axes, ejemplos):
    graficar_ejemplo(ax, v)
fig.suptitle('T09.4 — Interpolación acotada: antes (azul) y después (naranja) en el hueco puntual más largo',
             fontsize=12.5, color=COLOR_NAVY, fontweight='bold')
fig.tight_layout(); plt.show()

### T09.5 Vacíos prolongados: documentar y evaluar exclusión

Lo que **no** se pudo imputar (huecos prolongados y de borde) se trata con una regla explícita, en dos niveles:

1. **Exclusión de variable:** si tras imputar, el % de vacío de una variable supera su `max_pct_vacio`, se marca **EVALUAR EXCLUSIÓN DE VARIABLE**.
2. **Exclusión de bloque temporal:** una hora es **apta para modelado** solo si **todos los predictores** tienen dato (medido o imputado). Las horas con algún predictor vacío se agrupan en **bloques**, que se documentan con su causa probable:
   - **Falla de estación/datalogger:** casi todas las variables vacías a la vez, o diagnóstico también vacío.
   - **Falla de sensor individual:** solo algunas variables vacías.

Las filas **no se borran**: se marcan con `apto_modelado = False`. HU06 decidirá cómo construir rezagos sin cruzar estos bloques.

In [ ]:
# ==== T09.5 - Vacíos prolongados: bloques y exclusión ====
df_limpio['apto_modelado'] = df_limpio[PREDICTORES].notna().all(axis=1)
df_limpio['n_predictores_imputados'] = flags_imputacion[[f'{v}_imputado' for v in PREDICTORES]].sum(axis=1)

# --- Decisión por variable
decision_variables = []
for v, cfg in CONFIG_VARIABLES.items():
    pct_antes = 100 * df_crudo_h[v].isna().mean()
    pct_despues = 100 * df_limpio[v].isna().mean()
    exceso = pct_despues > cfg['max_pct_vacio']
    decision_variables.append({
        'variable': v, 'rol': cfg['rol'],
        'pct_vacio_antes': round(pct_antes, 2), 'pct_vacio_despues': round(pct_despues, 2),
        'completitud_despues_pct': round(100 - pct_despues, 2),
        'decision': 'EVALUAR EXCLUSIÓN DE VARIABLE' if exceso else 'CONSERVAR',
    })
decision_variables = pd.DataFrame(decision_variables)
display(decision_variables)

# --- Bloques temporales no aptos
no_apto = ~df_limpio['apto_modelado']
grupo = (no_apto != no_apto.shift(fill_value=False)).cumsum()
bloques = []
for _, b in df_limpio[no_apto].groupby(grupo[no_apto]):
    vacias = [v for v in PREDICTORES if df_limpio.loc[b.index, v].isna().all()]
    algun_vacio = [v for v in PREDICTORES if df_limpio.loc[b.index, v].isna().any()]
    diag_vacio = np.mean([df_limpio.loc[b.index, v].isna().mean() for v in DIAGNOSTICO]) if DIAGNOSTICO else 0
    if len(vacias) >= len(PREDICTORES) - 1 or diag_vacio >= 0.5:
        causa = 'Falla de estación / datalogger (vacío compartido)'
    else:
        causa = 'Falla de sensor individual'
    bloques.append({'inicio': b.index[0], 'fin': b.index[-1], 'horas': len(b),
                    'variables_afectadas': ', '.join(algun_vacio), 'causa_probable': causa,
                    'decision': 'Excluido del modelado (marcado apto_modelado=False)'})
bloques_excluidos = pd.DataFrame(bloques)
print(f'Bloques temporales no aptos: {len(bloques_excluidos)}  ·  horas no aptas: {int(no_apto.sum()):,} de {len(df_limpio):,} '
      f'({100 * no_apto.mean():.2f} %)')
if len(bloques_excluidos):
    display(bloques_excluidos.sort_values('horas', ascending=False).head(15).reset_index(drop=True))

### T09.6 Validación del tratamiento: completitud antes vs después

Criterio de aceptación de HU05: **completitud ≥ 90 %** o justificación del tratamiento. Se mide en dos niveles: **por variable** y **global** (horas con *todos* los predictores completos, que es lo realmente utilizable por el modelo).

In [ ]:
# ==== T09.6 - Completitud antes vs después ====
comp = decision_variables.set_index('variable')
completitud_global_antes = 100 * df_crudo_h[PREDICTORES].notna().all(axis=1).mean()
completitud_global_despues = 100 * df_limpio['apto_modelado'].mean()
print(f'Completitud GLOBAL (todos los predictores a la vez): antes = {completitud_global_antes:.2f} %  ->  después = {completitud_global_despues:.2f} %')
print(f'Criterio HU05: >= {UMBRAL_COMPLETITUD_PCT} %  ->  '
      f'{"CUMPLE" if completitud_global_despues >= UMBRAL_COMPLETITUD_PCT else "NO CUMPLE: justificar tratamiento (ver bloques prolongados)"}')

fig, ax = plt.subplots(figsize=(11, 4.6))
y = np.arange(len(comp))
ax.barh(y + 0.2, 100 - comp['pct_vacio_antes'], height=0.38, color=COLOR_GRAY, label='Antes')
ax.barh(y - 0.2, comp['completitud_despues_pct'], height=0.38, color=COLOR_TEAL, label='Después')
ax.axvline(UMBRAL_COMPLETITUD_PCT, color=COLOR_RED, ls='--', lw=1.4, label=f'Umbral {UMBRAL_COMPLETITUD_PCT} %')
ax.set_yticks(y); ax.set_yticklabels(comp.index, fontsize=9); ax.invert_yaxis()
ax.set_xlim(min(85, (100 - comp['pct_vacio_antes']).min() - 1), 100.3)
ax.set_xlabel('Completitud (%)'); ax.legend(frameon=False, fontsize=9, loc='lower left')
ax.spines[['top', 'right']].set_visible(False)
ax.set_title('T09.6 — Completitud por variable: antes vs después de la imputación acotada',
             fontsize=12, color=COLOR_NAVY, fontweight='bold')
fig.tight_layout(); plt.show()

In [ ]:
# ==== T09 - Control: la limpieza no distorsionó las distribuciones ====
filas = []
for v in VARIABLES:
    a, d = df_crudo_h[v].dropna(), df_limpio[v].dropna()
    filas.append({'variable': v, 'media_antes': a.mean(), 'media_despues': d.mean(),
                  'desv_antes': a.std(), 'desv_despues': d.std(),
                  'variacion_media_pct': 100 * (d.mean() - a.mean()) / a.mean() if a.mean() else np.nan})
control_dist = pd.DataFrame(filas).round(3)
display(control_dist)
print('Lectura: la media y la dispersión deben ser prácticamente iguales. Un cambio grande indicaría que se imputó demasiado.')

## 3.1.2 T10 — Corregir inconsistencias en horas de indisponibilidad

Las **horas de indisponibilidad** son la base de las dos variables objetivo (`horas_hasta_inicio_24h` y `duracion_indisponibilidad_h`). Un evento con fechas invertidas o una duración que no coincide con `fecha_fin − fecha_inicio` **contamina directamente las etiquetas**. Por eso T10 trabaja sobre el histórico de eventos, con reglas ordenadas y **una bitácora de cada cambio**.

### Reglas de T10 (en orden)

| # | Regla | Acción por defecto |
|---|---|---|
| R0 | Duplicados exactos y `id_evento` repetido | Conservar la **última** fila por `id_evento`; registrar si las filas repetidas **no** eran idénticas |
| R1 | `fecha_inicio` nula | Excluir (no hay evento definible) |
| R2 | `fecha_fin` nula | Según `POLITICA_FECHA_FIN_NULA`: por defecto `fin = inicio + duración` si `duracion_h_registrada` y `duracion_h_calculada` coinciden; si no, excluir |
| R3 | `fecha_fin < fecha_inicio` (fechas invertidas) | **Intercambiar** si eso hace coincidir con la duración registrada; si no, excluir y marcar para validar |
| R4 | `duracion_h_registrada` ≠ `fecha_fin − fecha_inicio` | La **fuente de verdad son las fechas** (de ahí salen los targets); se corrige la duración y se registra |
| R5 | Duración ≤ 0 | Excluir |
| R6 | Duración atípica (Q3 + 3·IQR) | **Solo marcar**; un evento largo puede ser real |
| R7 | Formato de `variable_dominante` | Normalizar (minúsculas, sin tildes, variantes unificadas) |
| R8 | Eventos solapados | Marcar (o fusionar si `FUSIONAR_SOLAPADOS=True`) |
| R9 | Causa no meteo-oceanográfica | Excluir según `causa_registrada` y `variable_dominante` (se cruzan ambas); si no hubiera columna de causa, queda pendiente de validar con Operaciones (D02/Q04) |

> **Aviso anti data-leakage (Q07):** las columnas `*_durante_evento_*` del histórico (oleaje/viento máximo, temperatura, humedad, presión durante el evento) son lecturas **posteriores a `t`**. Sirven para caracterizar y validar el evento (Rol B), **nunca como predictores**. HU06 no debe incluirlas en `X`.
>
> **Principio:** cada decisión que **no** es una corrección mecánica obvia queda marcada con `requiere_validacion = True`, para llevarla al Product Owner en lugar de decidirla en silencio.

In [ ]:
# ==== T10 - Preparación de la bitácora y utilidades ====
bitacora_eventos = []

def registrar(id_evento, regla, campo, antes, despues, accion, requiere_validacion=False):
    bitacora_eventos.append({'id_evento': id_evento, 'regla': regla, 'campo': campo, 'valor_antes': antes,
                             'valor_despues': despues, 'accion': accion, 'requiere_validacion': requiere_validacion})

def _norm_texto(x):
    if pd.isna(x): return x
    t = unicodedata.normalize('NFKD', str(x)).encode('ascii', 'ignore').decode('ascii')
    return re.sub(r'\s+', ' ', t).strip().lower()

ev = df_eventos_crudo.copy().reset_index(drop=True)
ev['_orden_original'] = np.arange(len(ev))
embudo = [('Eventos en el histórico crudo', len(ev))]

In [ ]:
# ==== R0 / R1 / R2 - Duplicados y nulos en eventos (parte T09 aplicada al histórico) ====
# R0a: filas completamente idénticas
n_ident = int(ev.drop(columns='_orden_original').duplicated().sum())
if n_ident:
    for i in ev[ev.drop(columns='_orden_original').duplicated(keep='last')]['id_evento']:
        registrar(i, 'R0', 'fila', 'duplicada exacta', 'eliminada', 'Se conserva una sola copia')
ev = ev.drop_duplicates(subset=[c for c in ev.columns if c != '_orden_original'], keep='last')

# R0b: id_evento repetido con contenido distinto -> se conserva la ÚLTIMA y se avisa del conflicto
dup_id = ev[ev.duplicated('id_evento', keep=False)].sort_values(['id_evento', '_orden_original'])
for i, g in dup_id.groupby('id_evento'):
    registrar(i, 'R0', 'id_evento', f'{len(g)} filas con distinto contenido', 'se conserva la última',
              'Dato digitado más de una vez con valores distintos', requiere_validacion=True)
n_dup_id = int(ev.duplicated('id_evento').sum())
ev = ev.sort_values('_orden_original').drop_duplicates('id_evento', keep='last')
embudo.append(('Tras eliminar duplicados (R0)', len(ev)))

# R1: fecha_inicio nula
m = ev['fecha_inicio'].isna()
for i in ev.loc[m, 'id_evento']:
    registrar(i, 'R1', 'fecha_inicio', 'nula', 'excluido', 'Sin inicio no se puede definir el evento')
n_ini_nulo = int(m.sum()); ev = ev[~m]

# R2: fecha_fin nula
m = ev['fecha_fin'].isna()
n_fin_nulo = int(m.sum())
if 'duracion_h_registrada' in ev.columns and POLITICA_FECHA_FIN_NULA == 'imputar_con_duracion_registrada':
    ok = m & ev['duracion_h_registrada'].gt(0)
    if 'duracion_h_calculada' in ev.columns:      # solo si las dos duraciones del CSV coinciden entre sí
        ok &= (ev['duracion_h_calculada'] - ev['duracion_h_registrada']).abs() <= TOL_DURACION_H
    for i, ini, d in ev.loc[ok, ['id_evento', 'fecha_inicio', 'duracion_h_registrada']].itertuples(index=False):
        registrar(i, 'R2', 'fecha_fin', 'nula', str(ini + pd.Timedelta(hours=d)), 'fin = inicio + duración (registrada = calculada)', True)
    ev.loc[ok, 'fecha_fin'] = ev.loc[ok, 'fecha_inicio'] + pd.to_timedelta(ev.loc[ok, 'duracion_h_registrada'], unit='h')
    m = ev['fecha_fin'].isna()
for i in ev.loc[m, 'id_evento']:
    registrar(i, 'R2', 'fecha_fin', 'nula', 'excluido', 'Evento en curso al exportar: no se conoce la duración real', True)
ev = ev[~m]
embudo.append(('Tras tratar fechas nulas (R1-R2)', len(ev)))

print(f'R0  duplicados exactos: {n_ident}  ·  id_evento repetidos: {n_dup_id}')
print(f'R1  fecha_inicio nula: {n_ini_nulo}  ·  R2 fecha_fin nula: {n_fin_nulo} (política: {POLITICA_FECHA_FIN_NULA})')

In [ ]:
# ==== R3 / R4 / R5 / R6 - Fechas invertidas y duración ====
ev = ev.copy()
tiene_dur_reg = 'duracion_h_registrada' in ev.columns
dur_fechas = (ev['fecha_fin'] - ev['fecha_inicio']) / pd.Timedelta(hours=1)

# R3: fechas invertidas
inv = dur_fechas < 0
n_swap = n_inv_excl = 0
for idx in ev.index[inv]:
    ini, fin = ev.at[idx, 'fecha_inicio'], ev.at[idx, 'fecha_fin']
    dur_swap = (ini - fin) / pd.Timedelta(hours=1)
    reg = ev.at[idx, 'duracion_h_registrada'] if tiene_dur_reg else np.nan
    if tiene_dur_reg and pd.notna(reg) and abs(dur_swap - reg) <= TOL_DURACION_H:
        ev.at[idx, 'fecha_inicio'], ev.at[idx, 'fecha_fin'] = fin, ini
        registrar(ev.at[idx, 'id_evento'], 'R3', 'fecha_inicio/fecha_fin', f'{ini} / {fin}', f'{fin} / {ini}',
                  'Fechas intercambiadas: coincide con la duración registrada')
        n_swap += 1
    else:
        registrar(ev.at[idx, 'id_evento'], 'R3', 'fecha_inicio/fecha_fin', f'{ini} / {fin}', 'excluido',
                  'Fechas invertidas y no coincide con la duración registrada: validar con Operaciones', True)
        ev.at[idx, '_excluir'] = True
if '_excluir' in ev.columns:
    n_inv_excl = int(ev['_excluir'].fillna(False).sum())
    ev = ev[~ev['_excluir'].fillna(False).astype(bool)].drop(columns='_excluir')
embudo.append(('Tras corregir fechas invertidas (R3)', len(ev)))

# R4: duración registrada vs fin - inicio (la fuente de verdad son las fechas)
ev['duracion_h'] = (ev['fecha_fin'] - ev['fecha_inicio']) / pd.Timedelta(hours=1)
n_dur_corr = n_dur_revisar = 0
if tiene_dur_reg:
    dif = (ev['duracion_h'] - ev['duracion_h_registrada']).abs()
    mal = dif > TOL_DURACION_H
    for idx in ev.index[mal | ev['duracion_h_registrada'].isna()]:
        reg, calc, d = ev.at[idx, 'duracion_h_registrada'], ev.at[idx, 'duracion_h'], dif.at[idx]
        revisar = bool(pd.notna(d) and d > UMBRAL_REVISION_H)
        registrar(ev.at[idx, 'id_evento'], 'R4', 'duracion_h', reg, round(calc, 3),
                  'Duración recalculada como fecha_fin - fecha_inicio', revisar)
        n_dur_corr += 1; n_dur_revisar += int(revisar)

# R5: duración <= 0
m = ev['duracion_h'] <= 0
for i in ev.loc[m, 'id_evento']:
    registrar(i, 'R5', 'duracion_h', '<= 0', 'excluido', 'Duración nula: no es un evento utilizable')
ev = ev[~m]
embudo.append(('Tras eliminar duraciones <= 0 (R5)', len(ev)))

# R6: duración atípica (solo marca)
q1, q3 = ev['duracion_h'].quantile([.25, .75]); lim_atip = q3 + 3 * (q3 - q1)
ev['duracion_atipica'] = ev['duracion_h'] > lim_atip
for i, d in ev.loc[ev['duracion_atipica'], ['id_evento', 'duracion_h']].itertuples(index=False):
    registrar(i, 'R6', 'duracion_h', round(d, 2), round(d, 2), f'Duración atípica (> {lim_atip:.1f} h): se conserva y se marca', True)

print(f'R3  fechas invertidas: intercambiadas = {n_swap}, excluidas = {n_inv_excl}')
if 'duracion_h_calculada' in ev.columns:
    print(f"R4  info: duracion_h_calculada (del CSV) difiere de fin-inicio en {int(((ev['duracion_h'] - ev['duracion_h_calculada']).abs() > TOL_DURACION_H).sum())} eventos; se recalcula siempre desde las fechas")
print(f'R4  duraciones corregidas: {n_dur_corr}  (de ellas {n_dur_revisar} con diferencia > {UMBRAL_REVISION_H} h -> validar)')
print(f'R6  duraciones atípicas marcadas (> {lim_atip:.1f} h): {int(ev["duracion_atipica"].sum())}')

In [ ]:
# ==== R7 / R8 / R9 - Formato, solapes y causa ====
# R7: normalizar variable_dominante
if 'variable_dominante' in ev.columns:
    antes = ev['variable_dominante'].nunique()
    base = ev['variable_dominante'].apply(_norm_texto)
    def _unificar(t):
        if pd.isna(t): return t
        if re.search(r'^no |no meteo', t):        return 'no_meteo_oceanografica'
        if re.search(r'ola|oleaj|swell|wave', t): return 'oleaje'
        if re.search(r'viento|wind', t):          return 'viento'
        return t
    ev['variable_dominante'] = base.apply(_unificar)
    despues = ev['variable_dominante'].nunique()
    print(f'R7  variantes de variable_dominante: {antes} -> {despues}')
    display(ev['variable_dominante'].value_counts(dropna=False).rename('eventos').to_frame())
    print('Revisar manualmente cualquier categoría distinta de viento/oleaje/no_meteo_oceanografica: no se unifica sin criterio de Operaciones.')

# R8: solapes
ev = ev.sort_values('fecha_inicio').reset_index(drop=True)
fin_max_previo = ev['fecha_fin'].cummax().shift()
ev['solapa_con_anterior'] = ev['fecha_inicio'] < fin_max_previo
n_solapes = int(ev['solapa_con_anterior'].sum())
if n_solapes and FUSIONAR_SOLAPADOS:
    fusion, actual = [], None
    for r in ev.to_dict('records'):
        if actual is not None and r['fecha_inicio'] < actual['fecha_fin']:
            registrar(r['id_evento'], 'R8', 'evento', 'solapado', f"fusionado con {actual['id_evento']}", 'Eventos fusionados', True)
            actual['fecha_fin'] = max(actual['fecha_fin'], r['fecha_fin'])
        else:
            if actual is not None: fusion.append(actual)
            actual = r
    fusion.append(actual)
    ev = pd.DataFrame(fusion)
    ev['duracion_h'] = (ev['fecha_fin'] - ev['fecha_inicio']) / pd.Timedelta(hours=1)
    ev['solapa_con_anterior'] = False
elif n_solapes:
    for i in ev.loc[ev['solapa_con_anterior'], 'id_evento']:
        registrar(i, 'R8', 'evento', 'solapa con el anterior', 'marcado', 'Solo marcado (FUSIONAR_SOLAPADOS=False)', True)
print(f'R8  eventos solapados: {n_solapes}  (fusionar = {FUSIONAR_SOLAPADOS})')

# R9: causa no meteo-oceanográfica
col_causa = next((c for c in COLUMNAS_CAUSA_CANDIDATAS if c in ev.columns), None)
n_no_clim = 0
if col_causa:
    texto = ev[col_causa].apply(_norm_texto).fillna('')
    no_clim_causa = texto.apply(lambda t: any(p in t for p in PALABRAS_CAUSA_NO_CLIMATICA))
    no_clim_dom = ev['variable_dominante'].eq('no_meteo_oceanografica') if 'variable_dominante' in ev.columns else pd.Series(False, index=ev.index)
    discrepancias = int((no_clim_causa != no_clim_dom).sum())
    no_clim = no_clim_causa | no_clim_dom
    print(f'    Cruce causa_registrada vs variable_dominante: {discrepancias} discrepancias (0 = ambas fuentes coinciden)')
    display(ev.assign(_causa=texto)[no_clim].groupby(col_causa).size().rename('eventos_excluidos').to_frame())
    for i, c in ev.loc[no_clim, ['id_evento', col_causa]].itertuples(index=False):
        registrar(i, 'R9', col_causa, c, 'excluido', 'Causa no meteo-oceanográfica', False)
    n_no_clim = int(no_clim.sum()); ev = ev[~no_clim]
    display(ev.get('variable_dominante', pd.Series(dtype=str)).value_counts().rename('eventos_conservados').to_frame())
    print(f'R9  columna de causa usada: {col_causa!r}  ->  eventos excluidos por causa no climática: {n_no_clim}')
else:
    print('R9  El histórico NO trae una columna de causa reconocible: la depuración de eventos no meteo-oceanográficos '
          'queda PENDIENTE de validar con Operaciones (control Q04 / decisión D02).')
embudo.append(('Eventos limpios finales', len(ev)))

# Cobertura de sensores (informativo: no se excluye)
cob_ini, cob_fin = df_limpio.index.min(), df_limpio.index.max()
ev['en_cobertura_sensores'] = (ev['fecha_fin'] >= cob_ini) & (ev['fecha_inicio'] <= cob_fin)
print(f'Eventos dentro del periodo con sensores ({cob_ini:%Y-%m-%d} a {cob_fin:%Y-%m-%d}): {int(ev["en_cobertura_sensores"].sum())} de {len(ev)}')

df_eventos_limpio = ev.drop(columns=['_orden_original'], errors='ignore').reset_index(drop=True)
bitacora_correcciones_eventos = pd.DataFrame(bitacora_eventos)

In [ ]:
# ==== Vista visual - Embudo de depuración de eventos + resumen de reglas ====
etq, val = zip(*embudo)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw={'width_ratios': [1.25, 1]})
ax = axes[0]
y = np.arange(len(etq))
ax.barh(y, val, color=[COLOR_GRAY] + [COLOR_TEAL] * (len(val) - 2) + [COLOR_NAVY])
for yi, v in zip(y, val): ax.text(v, yi, f' {v}', va='center', fontsize=9, color=COLOR_NAVY, fontweight='bold')
ax.set_yticks(y); ax.set_yticklabels(etq, fontsize=8.8); ax.invert_yaxis()
ax.spines[['top', 'right']].set_visible(False); ax.set_xlabel('N.° de eventos')
ax.set_title('Embudo de depuración del histórico', fontsize=11.5, color=COLOR_NAVY, fontweight='bold')

ax = axes[1]
if len(bitacora_correcciones_eventos):
    r = bitacora_correcciones_eventos.groupby('regla').size()
    ax.bar(r.index, r.values, color=COLOR_ORANGE)
    for xi, v in zip(r.index, r.values): ax.text(xi, v, str(v), ha='center', va='bottom', fontsize=9, fontweight='bold', color=COLOR_NAVY)
else:
    ax.text(0.5, 0.5, 'Sin correcciones registradas', ha='center', va='center', transform=ax.transAxes, color=COLOR_GRAY)
ax.spines[['top', 'right']].set_visible(False); ax.set_ylabel('N.° de registros en bitácora')
ax.set_title('Intervenciones por regla (T10)', fontsize=11.5, color=COLOR_NAVY, fontweight='bold')
fig.suptitle('T10 — Corrección de inconsistencias en horas de indisponibilidad', fontsize=13, color=COLOR_NAVY, fontweight='bold')
fig.tight_layout(); plt.show()

if len(bitacora_correcciones_eventos):
    pend = bitacora_correcciones_eventos[bitacora_correcciones_eventos['requiere_validacion']]
    print(f'Registros que requieren validación con Operaciones / Product Owner: {len(pend)}')
    display(pend.head(15))

### T10.1 Reconstrucción de los targets con los eventos limpios

Se vuelven a construir `ocurre_evento_24h`, `horas_hasta_inicio_24h` y `duracion_indisponibilidad_h` sobre la malla limpia. La función implementa **exactamente la definición de HU01/T02**: para cada origen `t` se toma el evento **más próximo** con inicio en `(t, t+24h]`; si no existe, los targets quedan en **NaN** (no se fuerza a 0, regla Q10).

> Si en tu sesión ya existe `construir_targets_evento()` del Sprint 1, puedes usarla: devuelve las mismas columnas. Aquí se incluye una versión **autocontenida y vectorizada** para que esta parte corra por sí sola.

In [ ]:
# ==== T10.1 - Targets con eventos limpios ====
def construir_targets_hu05(timestamps, df_eventos, horizonte_h=HORIZONTE_H):
    ev_ord = df_eventos.sort_values('fecha_inicio')
    inicios = ev_ord['fecha_inicio'].values.astype('datetime64[ns]')
    duraciones = ((ev_ord['fecha_fin'] - ev_ord['fecha_inicio']) / pd.Timedelta(hours=1)).values
    t = pd.DatetimeIndex(timestamps).values.astype('datetime64[ns]')
    pos = np.searchsorted(inicios, t, side='right')                 # primer inicio estrictamente > t
    hay = pos < len(inicios)
    pos_c = np.where(hay, pos, 0)
    h_hasta = (inicios[pos_c] - t) / np.timedelta64(1, 'h') if len(inicios) else np.full(len(t), np.nan)
    en_ventana = hay & (h_hasta <= horizonte_h)
    return pd.DataFrame({
        'ocurre_evento_24h': en_ventana.astype(int),
        'horas_hasta_inicio_24h': np.where(en_ventana, h_hasta, np.nan),
        'duracion_indisponibilidad_h': np.where(en_ventana, duraciones[pos_c], np.nan),
    }, index=pd.DatetimeIndex(timestamps))

# Comparación: filtro provisional de HU03 vs depuración completa de HU05
prov = (df_eventos_crudo.dropna(subset=['fecha_inicio', 'fecha_fin']).drop_duplicates('id_evento'))
prov = prov[prov['fecha_fin'] > prov['fecha_inicio']]
t_prov = construir_targets_hu05(df_limpio.index, prov)
t_limpio = construir_targets_hu05(df_limpio.index, df_eventos_limpio)

comparacion_targets = pd.DataFrame({
    'Filtro provisional (HU03)': [len(prov), int(t_prov['ocurre_evento_24h'].sum()), round(t_prov['duracion_indisponibilidad_h'].mean(), 2)],
    'Depuración completa (HU05)': [len(df_eventos_limpio), int(t_limpio['ocurre_evento_24h'].sum()), round(t_limpio['duracion_indisponibilidad_h'].mean(), 2)],
}, index=['Eventos utilizados', 'Horas con evento en las próximas 24 h', 'Duración media del evento (h)'])
display(comparacion_targets)

df_limpio = df_limpio.join(t_limpio)

## Verificaciones finales, exportación y Definition of Done

Se ejecutan **comprobaciones automáticas** sobre el dataset resultante y se exportan los entregables de HU05.

In [ ]:
# ==== Verificaciones finales (assertions) ====
chequeos = {
    'Índice horario único y ordenado': df_limpio.index.is_unique and df_limpio.index.is_monotonic_increasing,
    'Malla horaria continua (sin saltos)': (df_limpio.index.to_series().diff().dropna() == pd.Timedelta(hours=1)).all(),
    'Sin valores fuera de rango físico en ninguna variable': all(
        df_limpio[v].dropna().between(*CONFIG_VARIABLES[v]['rango']).all() for v in VARIABLES),
    'Imputación solo en horas originalmente vacías': all(
        df_rango.loc[flags_imputacion[f'{v}_imputado'], v].isna().all() for v in VARIABLES),
    'Eventos: id_evento único': df_eventos_limpio['id_evento'].is_unique,
    'Eventos: fecha_fin > fecha_inicio en todos': (df_eventos_limpio['fecha_fin'] > df_eventos_limpio['fecha_inicio']).all(),
    'Eventos: duracion_h = fin - inicio': ((df_eventos_limpio['fecha_fin'] - df_eventos_limpio['fecha_inicio']) / pd.Timedelta(hours=1) - df_eventos_limpio['duracion_h']).abs().max() < 1e-6,
    'Targets: horas_hasta_inicio_24h dentro de (0, 24]': t_limpio['horas_hasta_inicio_24h'].dropna().between(0, HORIZONTE_H, inclusive='right').all(),
    'Targets: sin 0 forzado donde no hay evento (NaN)': t_limpio.loc[t_limpio['ocurre_evento_24h'] == 0, 'horas_hasta_inicio_24h'].isna().all(),
}
# Verificación estricta de huecos imputados: ninguno más largo que el límite
for v in VARIABLES:
    m = flags_imputacion[f'{v}_imputado']
    g = (m != m.shift(fill_value=False)).cumsum()
    mx = int(m.groupby(g).sum().max()) if m.any() else 0
    chequeos[f'Máx. racha imputada {v} <= {CONFIG_VARIABLES[v]["max_gap_h"]} h'] = mx <= CONFIG_VARIABLES[v]['max_gap_h']

tabla_chequeos = pd.DataFrame({'Verificación': list(chequeos), 'Resultado': ['OK' if bool(v) else 'FALLA' for v in chequeos.values()]})
display(tabla_chequeos)
assert (tabla_chequeos['Resultado'] == 'OK').all(), 'Hay verificaciones que fallan: revisar la tabla anterior.'

In [ ]:
# ==== Exportación de entregables de HU05 ====
columnas_modelo = (['velocidad_viento_mph'] if 'velocidad_viento_mph' in df_limpio.columns else []) + VARIABLES
columnas_modelo = list(dict.fromkeys(columnas_modelo))
dataset_limpio = df_limpio[columnas_modelo + ['apto_modelado', 'n_predictores_imputados',
                                              'ocurre_evento_24h', 'horas_hasta_inicio_24h', 'duracion_indisponibilidad_h']]

dataset_limpio.to_csv('dataset_limpio_horario_HU05.csv')
flags_imputacion.to_csv('flags_imputacion_HU05.csv')
df_eventos_limpio.to_csv('eventos_limpios_HU05.csv', index=False)
bitacora_vacios.to_csv('bitacora_vacios_HU05.csv', index=False)
bitacora_correcciones_eventos.to_csv('bitacora_correcciones_eventos_HU05.csv', index=False)
decision_variables.to_csv('decision_variables_HU05.csv', index=False)

print('Archivos generados:')
for f in ['dataset_limpio_horario_HU05.csv', 'flags_imputacion_HU05.csv', 'eventos_limpios_HU05.csv',
          'bitacora_vacios_HU05.csv', 'bitacora_correcciones_eventos_HU05.csv', 'decision_variables_HU05.csv']:
    print(f'  - {f}  ({Path(f).stat().st_size/1024:,.0f} KB)')

try:
    from google.colab import files
    # files.download('dataset_limpio_horario_HU05.csv')   # descomentar para descargar
except ImportError:
    pass

### Definition of Done — HU05 (T09 / T10)

La tabla se **calcula con los resultados reales** de esta ejecución (no está escrita a mano).

In [ ]:
# ==== HU05 - Definition of Done (calculada) ====
def estado(c): return 'CUMPLE' if c else 'NO CUMPLE'
n_vars_exclusion = int((decision_variables['decision'] != 'CONSERVAR').sum())
dod_hu05 = pd.DataFrame([
    ['T09', 'Verificación y deduplicación por marca de tiempo documentada (CSV crudos e índice horario)', 'CUMPLE'],
    ['T09', 'Vacíos puntuales imputados con interpolación lineal/spline/circular acotada por variable', estado((tabla_chequeos['Resultado'] == 'OK').all())],
    ['T09', 'Vacíos prolongados documentados en bitácora, sin imputar', 'CUMPLE'],
    ['T09', f'Completitud global >= {UMBRAL_COMPLETITUD_PCT} % (actual: {completitud_global_despues:.2f} %)', estado(completitud_global_despues >= UMBRAL_COMPLETITUD_PCT)],
    ['T09', f'Variables a evaluar para exclusión: {n_vars_exclusion}', 'CUMPLE' if n_vars_exclusion == 0 else 'REVISAR'],
    ['T09', 'Banderas de imputación exportadas (dato medido vs imputado)', 'CUMPLE'],
    ['T10', 'Fechas invertidas y duraciones inconsistentes corregidas con bitácora', 'CUMPLE'],
    ['T10', 'Targets reconstruidos con eventos limpios', 'CUMPLE'],
    ['T10', 'Depuración de eventos por causa no meteo-oceanográfica',
     'CUMPLE' if col_causa else 'PENDIENTE (validar con Operaciones: no hay columna de causa)'],
    ['-', 'Decisiones que requieren validación del Product Owner listadas en la bitácora',
     f'{int(bitacora_correcciones_eventos["requiere_validacion"].sum()) if len(bitacora_correcciones_eventos) else 0} registros'],
], columns=['Tarea', 'Criterio de aceptación', 'Estado'])
display(dod_hu05)

### Decisiones abiertas para validar (llevar a la asesora / Product Owner)

1. **Límites de interpolación por variable** (`max_gap_h`): son valores de partida; confirmar o ajustar con el diagnóstico de T09.3 y el criterio de Operaciones.
2. **`fecha_fin` nula:** se imputa como inicio + duración cuando las dos duraciones del CSV coinciden; confirmar con Operaciones (la alternativa es excluir el evento).
3. **Fuente de verdad de la duración:** se asumen las **fechas** (de ahí salen los targets); confirmar que la duración digitada no es la correcta en los casos con diferencia > 1 h.
4. **Eventos de causa no meteo-oceanográfica:** requieren una columna de causa en el histórico, o la validación directa con Operaciones (D02/Q04).
5. **Solapes entre eventos:** por defecto solo se marcan; decidir si se fusionan.

### Puente hacia HU06 (Ingeniería de características)

HU06 debe **respetar `apto_modelado`** y las banderas de imputación al construir los rezagos (t-1, t-2, t-3, t-6, t-12, t-24) y las ventanas móviles de 6 y 24 h: una ventana que cruza un bloque excluido no debe tratarse como dato continuo.